# DeepForm — Correction automatique d'examens semi-structurés
### Projet Computer Vision · IG.2405 – 2026

Ce notebook est **autonome** : tout le code est inclus ci-dessous, il n'y a
**aucun `git clone`** à faire. Vous fournissez uniquement vos données
(un fichier `.zip` ou un dossier Google Drive) puis vous exécutez les
cellules de haut en bas.

**Pipeline :**
- **Programme 1 — `autoValidPresences`** : à partir des photos de 1ʳᵉ page,
  lit le `StudentID` sur la grille à bulles (méthodes bas-niveau) et
  authentifie la signature (vérification 1:1) → `EXAM_FORMXX_PRESENCES.xlsx`.
- **Programme 2 — `autoReadForm`** : à partir des PDF scannés, lit la page
  d'identification (PAGE-01) et les réponses QCM/numériques (onglet EXAM)
  → un `.xlsx` par PDF.

**Contrainte cahier des charges (§4.1) :** les éléments **graphiques**
(grilles, cases à cocher, signatures, cryptogrammes) sont traités uniquement
par des méthodes **bas-niveau** (filtrage, Hough, morphologie, rotation).
Seuls les **textes** imprimés/manuscrits utilisent l'OCR / un réseau de
neurones.


## Étape 1 — Dépendances système et Python

In [ ]:
# Poppler (PDF→image), Tesseract (OCR), libheif (photos HEIC)
!apt-get -qq install -y poppler-utils tesseract-ocr tesseract-ocr-fra libheif1 >/dev/null
!pip -q install opencv-python-headless pdf2image pytesseract openpyxl pillow-heif numpy >/dev/null
print("Dépendances installées.")


## Étape 2 — Recréer l'arborescence du code (autonome, sans Git)

Chaque cellule `%%writefile` écrit un module sur le disque Colab. Exécutez-les
toutes une fois ; les imports fonctionneront ensuite normalement.


In [ ]:
import os
os.makedirs('utils', exist_ok=True)
print('Dossier utils/ prêt.')

In [ ]:
%%writefile utils/__init__.py
# Vision par ordinateur - Utilitaires


In [ ]:
%%writefile utils/image_processing.py
"""Low-level image processing utilities (filtering, Hough, morphology, rotation)."""

import cv2
import numpy as np


def load_image(path):
    """Load image as grayscale and color. Handles HEIC, uppercase extensions, etc."""
    path = str(path)

    # 1. Standard cv2 load
    img_color = cv2.imread(path)

    # 2. np.fromfile fallback (handles uppercase extensions on Linux)
    if img_color is None:
        try:
            raw = np.fromfile(path, dtype=np.uint8)
            img_color = cv2.imdecode(raw, cv2.IMREAD_COLOR)
        except Exception:
            img_color = None

    # 3. Pillow fallback — covers HEIC/HEIF (pillow-heif), WebP, TIFF, etc.
    if img_color is None:
        try:
            from PIL import Image as _PILImage
            # Register HEIC support if pillow-heif is available
            try:
                import pillow_heif
                pillow_heif.register_heif_opener()
            except ImportError:
                pass
            pil_img = _PILImage.open(path).convert("RGB")
            img_color = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        except Exception:
            img_color = None

    if img_color is None:
        raise FileNotFoundError(f"Cannot load image: {path}")

    img_gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
    return img_color, img_gray


def preprocess(gray):
    """Denoise and binarize a grayscale image."""
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary


def _order_points(pts):
    """Order 4 corners: top-left, top-right, bottom-right, bottom-left."""
    rect = np.zeros((4, 2), dtype=np.float32)
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1).ravel()
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect


def _find_corner_mark(roi, expect_corner):
    """
    Find the centroid of the L-bracket registration mark in a corner ROI.
    expect_corner: 'tl', 'tr', 'bl', 'br'
    Returns (cx, cy) in ROI coordinates, or None.

    An L-bracket is a thin L-shaped stroke: its bounding box is roughly square
    but only sparsely filled. Text (e.g. "Module") and solid marks are dense, so
    we reject candidates that are too dense, too small, too big, or too elongated
    — then pick the bracket closest to the expected corner.
    """
    rh, rw = roi.shape
    short = min(rh, rw)

    _, bw = cv2.threshold(roi, 90, 255, cv2.THRESH_BINARY_INV)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel)
    cnts, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None

    corners = {'tl': (0, 0), 'tr': (rw, 0), 'bl': (0, rh), 'br': (rw, rh)}
    ex, ey = corners[expect_corner]

    best, best_d = None, float('inf')
    for cnt in cnts:
        bx, by, bw2, bh2 = cv2.boundingRect(cnt)
        max_dim = max(bw2, bh2)
        min_dim = min(bw2, bh2)
        # Size: the L-bracket spans a noticeable fraction of the ROI, but is not
        # the whole banner/border.
        if max_dim < short * 0.06 or max_dim > short * 0.6:
            continue
        # Shape: roughly square (an L fits in a ~square box, not a thin line).
        if min_dim / max_dim < 0.4:
            continue
        # Sparsity: an L outline fills little of its bbox; dense blobs are text
        # or solid squares.
        fill = cv2.contourArea(cnt) / (bw2 * bh2 + 1e-6)
        if fill > 0.55:
            continue
        cx = bx + bw2 // 2
        cy = by + bh2 // 2
        d = (cx - ex) ** 2 + (cy - ey) ** 2
        if d < best_d:
            best_d = d
            best = (cx, cy)
    return best


def correct_perspective(img_gray):
    """
    Use the 4 L-bracket registration marks to warp the camera photo into the same
    coordinate system as the scanned PDF, so all calibrated relative coordinates
    (STUDENT_ID_REGION, GROUP_DIGITS_REGION, etc.) apply directly.

    PDF-measured L-bracket positions (relative to page):
        tl=(0.0798, 0.0588)  tr=(0.8690, 0.1149)
        bl=(0.1001, 0.9320)  br=(0.9018, 0.9306)

    We map the detected marks in the photo to those exact positions in the output
    image (A4 proportions, 1200 px wide), giving a warped image whose coordinate
    system matches the PDF exactly.

    Returns (corrected_img, success: bool).
    """
    # Known PDF-relative positions of the L-bracket centres (tl, tr, br, bl)
    PDF_MARKS = {
        'tl': (0.0798, 0.0588),
        'tr': (0.8690, 0.1149),
        'br': (0.9018, 0.9306),
        'bl': (0.1001, 0.9320),
    }

    h, w = img_gray.shape
    margin_x = int(w * 0.18)
    margin_y = int(h * 0.18)

    rois = {
        'tl': (img_gray[:margin_y, :margin_x],            0,           0),
        'tr': (img_gray[:margin_y, w - margin_x:],         w - margin_x, 0),
        'bl': (img_gray[h - margin_y:, :margin_x],         0,           h - margin_y),
        'br': (img_gray[h - margin_y:, w - margin_x:],     w - margin_x, h - margin_y),
    }

    src_pts = []
    for key in ('tl', 'tr', 'br', 'bl'):
        roi, ox, oy = rois[key]
        pt = _find_corner_mark(roi, key)
        if pt is None:
            return img_gray, False
        src_pts.append([pt[0] + ox, pt[1] + oy])

    src = np.array(src_pts, dtype=np.float32)

    tgt_w = 1200
    tgt_h = int(tgt_w * 297 / 210)   # A4 aspect ratio ≈ 1697

    dst = np.array(
        [[PDF_MARKS[k][0] * tgt_w, PDF_MARKS[k][1] * tgt_h]
         for k in ('tl', 'tr', 'br', 'bl')],
        dtype=np.float32,
    )

    M = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(img_gray, M, (tgt_w, tgt_h),
                                 flags=cv2.INTER_LINEAR,
                                 borderMode=cv2.BORDER_REPLICATE)
    return warped, True


def deskew(gray):
    """Correct document skew using Hough lines. Returns (deskewed_image, angle_degrees)."""
    binary = preprocess(gray)
    edges = cv2.Canny(binary, 50, 150, apertureSize=3)
    lines = cv2.HoughLines(edges, 1, np.pi / 180, threshold=200)

    angle = 0.0
    if lines is not None:
        angles = []
        for rho, theta in lines[:, 0]:
            a = np.degrees(theta) - 90
            if abs(a) < 45:
                angles.append(a)
        if angles:
            angle = np.median(angles)

    h, w = gray.shape
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    deskewed = cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_LINEAR,
                              borderMode=cv2.BORDER_REPLICATE)
    return deskewed, angle


def morpho_open(binary, ksize=3):
    """Morphological opening to remove small noise."""
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (ksize, ksize))
    return cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)


def morpho_close(binary, ksize=5):
    """Morphological closing to fill small holes."""
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (ksize, ksize))
    return cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)


def find_horizontal_lines(binary, min_len_ratio=0.3):
    """Detect horizontal lines via morphological erosion. Returns a binary mask."""
    h, w = binary.shape
    min_len = int(w * min_len_ratio)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (min_len, 1))
    inv = cv2.bitwise_not(binary)
    eroded = cv2.erode(inv, kernel)
    dilated = cv2.dilate(eroded, kernel)
    return dilated


def find_vertical_lines(binary, min_len_ratio=0.3):
    """Detect vertical lines via morphological erosion. Returns a binary mask."""
    h, w = binary.shape
    min_len = int(h * min_len_ratio)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, min_len))
    inv = cv2.bitwise_not(binary)
    eroded = cv2.erode(inv, kernel)
    dilated = cv2.dilate(eroded, kernel)
    return dilated


def detect_grid_cells(binary, n_rows, n_cols, region=None):
    """
    Detect filled bubble cells. Primary: contour detection + k-means clustering.
    Fallback: equal-grid division with per-column argmax.
    """
    if region is not None:
        x0, y0, rw, rh = region
        roi = binary[y0:y0+rh, x0:x0+rw]
    else:
        roi = binary

    roi_h, roi_w = roi.shape
    inv = cv2.bitwise_not(roi)

    exp_h = roi_h / n_rows
    exp_w = roi_w / n_cols

    # Find square-like bubble contours, strict size filter to exclude label text
    cnts, _ = cv2.findContours(inv.copy(), cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    seen = set()
    candidates = []
    for cnt in cnts:
        bx, by, bw, bh_ = cv2.boundingRect(cnt)
        if not (exp_w * 0.35 < bw < exp_w * 1.8 and
                exp_h * 0.35 < bh_ < exp_h * 1.8):
            continue
        if not (0.5 < bw / max(bh_, 1) < 2.0):
            continue
        key = (bx // 15, by // 15)
        if key in seen:
            continue
        seen.add(key)
        cx, cy = bx + bw // 2, by + bh_ // 2
        pad = max(2, int(min(bw, bh_) * 0.15))
        inner = inv[by + pad: by + bh_ - pad, bx + pad: bx + bw - pad]
        fill = np.sum(inner > 0) / max(inner.size, 1)
        candidates.append((cx, cy, fill))

    grid = np.zeros((n_rows, n_cols), dtype=bool)

    if len(candidates) < n_cols:
        # Fallback: equal-grid with per-column argmax
        cell_h = roi_h / n_rows
        cell_w = roi_w / n_cols
        fill_scores = np.zeros((n_rows, n_cols), dtype=float)
        for r in range(n_rows):
            for c in range(n_cols):
                r0, r1 = int(r * cell_h), int((r + 1) * cell_h)
                c0, c1 = int(c * cell_w), int((c + 1) * cell_w)
                py = max(1, (r1 - r0) // 6)
                px = max(1, (c1 - c0) // 6)
                cell = inv[r0 + py: r1 - py, c0 + px: c1 - px]
                fill_scores[r, c] = np.sum(cell > 0) / max(cell.size, 1)
        for c in range(n_cols):
            col = fill_scores[:, c]
            idx = int(np.argmax(col))
            if col[idx] > 0.03 and col[idx] > np.median(col) * 1.4:
                grid[idx, c] = True
        return grid

    pts_x = np.array([[c[0]] for c in candidates], dtype=np.float32)
    pts_y = np.array([[c[1]] for c in candidates], dtype=np.float32)
    n_c = min(n_cols, len(candidates))
    n_r = min(n_rows, len(candidates))

    _, col_lbls, col_centers = cv2.kmeans(
        pts_x, n_c, None,
        (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_MAX_ITER, 50, 1.0),
        10, cv2.KMEANS_PP_CENTERS)
    _, row_lbls, row_centers = cv2.kmeans(
        pts_y, n_r, None,
        (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_MAX_ITER, 50, 1.0),
        10, cv2.KMEANS_PP_CENTERS)

    col_rank = np.empty(n_c, dtype=int)
    col_rank[np.argsort(col_centers.flatten())] = np.arange(n_c)
    row_rank = np.empty(n_r, dtype=int)
    row_rank[np.argsort(row_centers.flatten())] = np.arange(n_r)

    fill_scores = np.zeros((n_rows, n_cols), dtype=float)
    for i, (cx, cy, fill) in enumerate(candidates):
        c = int(col_rank[col_lbls[i, 0]])
        r = int(row_rank[row_lbls[i, 0]])
        if 0 <= r < n_rows and 0 <= c < n_cols:
            fill_scores[r, c] = max(fill_scores[r, c], fill)

    for c in range(n_cols):
        col = fill_scores[:, c]
        idx = int(np.argmax(col))
        if col[idx] > 0.04 and col[idx] > np.median(col) * 1.5:
            grid[idx, c] = True

    return grid


def crop_region(img, x, y, w, h):
    """Crop a rectangular region from an image."""
    return img[y:y+h, x:x+w]


def normalize_signature(sig_gray, target_size=(128, 64)):
    """
    Normalize a signature image for matching.
    Uses adaptive thresholding (robust to camera lighting) + tight crop + resize.
    """
    # Adaptive threshold handles uneven illumination from phone photos
    binary = cv2.adaptiveThreshold(
        sig_gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV,
        blockSize=25, C=10)
    # Remove salt & pepper noise
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

    coords = cv2.findNonZero(binary)
    if coords is None:
        return np.zeros((target_size[1], target_size[0]), dtype=np.uint8)
    x, y, w, h = cv2.boundingRect(coords)
    pad = 4
    x, y = max(0, x - pad), max(0, y - pad)
    w = min(binary.shape[1] - x, w + 2 * pad)
    h = min(binary.shape[0] - y, h + 2 * pad)
    cropped = binary[y:y + h, x:x + w]
    return cv2.resize(cropped, target_size, interpolation=cv2.INTER_AREA)


def image_similarity(img_a, img_b):
    """Cosine similarity between two binary images. Returns score in [0, 1].
    Used for cryptogram comparison."""
    if img_a.shape != img_b.shape:
        img_b = cv2.resize(img_b, (img_a.shape[1], img_a.shape[0]),
                           interpolation=cv2.INTER_AREA)
    a = img_a.astype(np.float32) / 255.0
    b = img_b.astype(np.float32) / 255.0
    num = np.sum(a * b)
    den = np.sqrt(np.sum(a ** 2) * np.sum(b ** 2))
    return float(num / den) if den > 1e-8 else 0.0


def ncc_similarity(img_a, img_b):
    """Zero-mean NCC between two images. Returns score in [-1, 1].
    Used for signature matching."""
    if img_a.shape != img_b.shape:
        img_b = cv2.resize(img_b, (img_a.shape[1], img_a.shape[0]),
                           interpolation=cv2.INTER_AREA)
    a = img_a.astype(np.float32)
    b = img_b.astype(np.float32)
    a -= a.mean()
    b -= b.mean()
    num = np.sum(a * b)
    den = np.sqrt(np.sum(a ** 2) * np.sum(b ** 2))
    return float(num / den) if den > 1e-8 else 0.0


In [ ]:
%%writefile utils/pdf_utils.py
"""PDF-to-image conversion using pdf2image (poppler backend)."""

import numpy as np
import cv2
from pdf2image import convert_from_path


PDF_DPI = 250    # 250 DPI needed for reliable OCR of faint handwriting


def pdf_to_images(pdf_path):
    """Convert every page of a PDF to a grayscale numpy array. Returns list of uint8 arrays."""
    pil_pages = convert_from_path(str(pdf_path), dpi=PDF_DPI)
    pages = []
    for pil_img in pil_pages:
        rgb = np.array(pil_img)
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        pages.append(gray)
    return pages


In [ ]:
%%writefile utils/ocr_reader.py
"""
OCR utilities for printed and handwritten text.
Printed text uses Tesseract, handwritten digits use a small CNN (falls back to Tesseract).
"""

import cv2
import numpy as np

try:
    import pytesseract
    _TESSERACT_AVAILABLE = True
except ImportError:
    _TESSERACT_AVAILABLE = False

try:
    import torch
    import torch.nn as nn
    _TORCH_AVAILABLE = True
except ImportError:
    _TORCH_AVAILABLE = False


def read_printed_text(img_gray, config="--psm 6"):
    """Run Tesseract on a grayscale crop and return stripped text."""
    if not _TESSERACT_AVAILABLE:
        return ""
    # light binarization helps Tesseract
    _, binary = cv2.threshold(img_gray, 0, 255,
                               cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    text = pytesseract.image_to_string(binary, config=config)
    return text.strip()


def read_printed_date(img_gray):
    """Read a date field (DD/MM/YYYY)."""
    return read_printed_text(img_gray,
                             config="--psm 7 -c tessedit_char_whitelist=0123456789/")


def read_printed_field(img_gray):
    """Read a single-line printed field."""
    return read_printed_text(img_gray, config="--psm 7")


class _DigitCNN(object if not _TORCH_AVAILABLE else nn.Module):
    """Minimal LeNet-5 style CNN for single handwritten digit recognition."""

    def __init__(self):
        if not _TORCH_AVAILABLE:
            return
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


_digit_model = None
_digit_model_path = "models/digit_cnn.pth"


def _get_digit_model():
    global _digit_model
    if _digit_model is not None:
        return _digit_model
    if not _TORCH_AVAILABLE:
        return None
    import torch
    model = _DigitCNN()
    try:
        state = torch.load(_digit_model_path, map_location="cpu")
        model.load_state_dict(state)
        model.eval()
        _digit_model = model
    except FileNotFoundError:
        # model not trained yet, fall back to Tesseract
        _digit_model = None
    return _digit_model


def read_handwritten_digit(img_gray):
    """Recognize a single handwritten digit (28x28-compatible crop). Returns 0-9 or -1 on failure."""
    model = _get_digit_model()
    if model is not None:
        import torch
        resized = cv2.resize(img_gray, (28, 28), interpolation=cv2.INTER_AREA)
        _, binary = cv2.threshold(resized, 0, 255,
                                   cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        tensor = torch.tensor(binary, dtype=torch.float32).unsqueeze(0).unsqueeze(0) / 255.0
        with torch.no_grad():
            logits = model(tensor)
        return int(logits.argmax(dim=1).item())

    # Tesseract fallback
    if _TESSERACT_AVAILABLE:
        text = read_printed_text(img_gray,
                                  config="--psm 10 -c tessedit_char_whitelist=0123456789")
        try:
            return int(text.strip())
        except ValueError:
            return -1
    return -1


def read_handwritten_text(img_gray):
    """Read free-form handwritten text (name, first name, etc.) using Tesseract."""
    if not _TESSERACT_AVAILABLE:
        return ""
    # scale up for better recognition
    h, w = img_gray.shape
    scale = max(1, 60 // h)
    resized = cv2.resize(img_gray, (w * scale, h * scale),
                         interpolation=cv2.INTER_CUBIC)
    return read_printed_text(resized, config="--psm 7")


def read_handwritten_number(img_gray):
    """Read a handwritten number (mantissa/exponent or integer). Returns raw string."""
    if not _TESSERACT_AVAILABLE:
        return ""
    _, binary = cv2.threshold(img_gray, 0, 255,
                               cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    text = pytesseract.image_to_string(
        binary,
        config="--psm 7 -c tessedit_char_whitelist=0123456789.,-Ee "
    )
    return text.strip()


In [ ]:
%%writefile utils/form_layout.py
"""
Layout descriptors for the exam form pages.
Coordinates are (x_ratio, y_ratio, w_ratio, h_ratio) as fractions of page width/height,
so they stay resolution-independent. Calibrate values against the actual form if needed.
"""

# Page 1 fields

PAGE1_FIELDS = {
    # CODES EXAM row — calibrated from PDF at 250 DPI
    # Module value "IG.1103" at x_ratio 0.193-0.290, y_ratio 0.108-0.125
    "module":     (0.193, 0.108, 0.10, 0.017),
    "professor":  (0.387, 0.108, 0.10, 0.017),
    "date":       (0.604, 0.108, 0.10, 0.017),
    "code":       (0.822, 0.108, 0.10, 0.017),
    # Exam conditions checkboxes — in the "Are authorised" section
    "notes_cours":       (0.04, 0.555, 0.08, 0.030),
    "notes_manuscrites": (0.19, 0.555, 0.08, 0.030),
    "ordinateur":        (0.35, 0.555, 0.08, 0.030),
    "calculatrice":      (0.51, 0.555, 0.08, 0.030),
    "feuilles_brouillon":     (0.66, 0.555, 0.08, 0.030),
    "feuilles_brouillon_nb":  (0.20, 0.585, 0.10, 0.025),
    # Note maximale "10" / Note pour valider "04" — calibrated from PDF.
    # maximale height tightened to exclude the box's bottom border line.
    "note_maximale": (0.556, 0.702, 0.145, 0.030),
    "note_valider":  (0.556, 0.742, 0.145, 0.028),
    # row 13 – Prénom (handwritten boxes)
    "prenom": (0.03, 0.155, 0.32, 0.045),
    # row 14 – Nom (handwritten boxes)
    "nom":    (0.03, 0.215, 0.32, 0.045),
    # row 15 – Signature zone
    "signature": (0.03, 0.255, 0.30, 0.245),
    # row 16 – Group (from bubble grid)
    "group":     (0.37, 0.19, 0.24, 0.36),
    # row 17 – StudentID (from bubble grid)
    "student_id": (0.75, 0.19, 0.22, 0.36),
    # row 18 – Cryptogram (bottom left)
    "cryptogram": (0.08, 0.935, 0.10, 0.05),
}

# Exam page fields

# Number of answer choices per question (A-H = 8 max, adapt to the actual form)
N_CHOICES = 8
CHOICE_LABELS = list("ABCDEFGH")[:N_CHOICES]

# Relative position of the answer grid on an exam page
EXAM_QUESTION_COL = (0.02, 0.10, 0.08, 0.80)
EXAM_CHOICES_START_X = 0.12
EXAM_CHOICE_COL_W = 0.07
EXAM_ROW_START_Y = 0.10
EXAM_ROW_H = 0.06

# Handwritten numerical answer columns
EXAM_MANTISSE_COL  = (0.70, 0.10, 0.15, 0.80)
EXAM_EXPOSANT_COL  = (0.86, 0.10, 0.08, 0.80)
EXAM_UNITE_COL     = (0.95, 0.10, 0.04, 0.80)


def field_to_pixels(rel_coords, page_h, page_w):
    """Convert relative (x,y,w,h) to integer pixel coordinates."""
    xr, yr, wr, hr = rel_coords
    return (int(xr * page_w), int(yr * page_h),
            int(wr * page_w), int(hr * page_h))


def crop_field(page_gray, rel_coords):
    """Crop a field from a page using relative coordinates."""
    h, w = page_gray.shape
    x, y, fw, fh = field_to_pixels(rel_coords, h, w)
    return page_gray[y:y+fh, x:x+fw]


In [ ]:
%%writefile utils/grid_reader.py
"""
Extracts student ID and group from the bubble grid on page 1.
Each column encodes one digit/letter (bubbles 0-9 top to bottom).

Coordinate system: fractions of image width/height.
The presence photos are camera shots of a printed form, so coordinates
are approximate; the grid detector uses robust contour-based methods.
"""

import cv2
import numpy as np
from utils.image_processing import preprocess, morpho_open


# ── Region definitions (fraction of image width/height) ─────────────────────
# These cover the A4 form as seen in a roughly-centered camera photo.

STUDENT_ID_REGION = (0.73, 0.18, 0.24, 0.38)   # 5-digit ID: 5 cols × 10 rows
STUDENT_ID_DIGITS = 5
STUDENT_ID_ROWS   = 10   # rows 0-9

# Group grid: precisely located via projection. Header excluded.
# Digit col0 center x≈0.562, col1 x≈0.590, letter x≈0.647.
# Rows 0-9 span y≈0.222 (top of row0) to y≈0.458 (bottom of row9).
GROUP_DIGITS_REGION = (0.548, 0.221, 0.056, 0.237)  # 2 digit columns
GROUP_LETTER_REGION = (0.632, 0.221, 0.030, 0.237)  # 1 letter column (A-J)
GROUP_ROWS     = 10

SIGNATURE_REGION = (0.02, 0.17, 0.85, 0.42)  # search area containing signature box
# ─────────────────────────────────────────────────────────────────────────────


def _locate_grid(page_gray, rel_region):
    """Convert relative region to absolute pixel coordinates."""
    h, w = page_gray.shape
    x  = int(rel_region[0] * w)
    y  = int(rel_region[1] * h)
    bw = int(rel_region[2] * w)
    bh = int(rel_region[3] * h)
    return x, y, bw, bh


def read_bubble_column(grid_bool, col):
    """Return the filled row index (0-9) for a column, or -1 if ambiguous/empty."""
    filled = [r for r in range(grid_bool.shape[0]) if grid_bool[r, col]]
    return filled[0] if len(filled) == 1 else -1


def _grid_to_string(grid, n_cols, letter_col=None):
    """Convert a boolean grid to a string of digits/letters."""
    chars = []
    for col in range(n_cols):
        d = read_bubble_column(grid, col)
        if d < 0:
            chars.append("?")
        elif letter_col is not None and col == letter_col:
            chars.append(chr(ord('A') + d))
        else:
            chars.append(str(d))
    return "".join(chars)


def extract_student_id(page_gray):
    """
    Extract the numeric student ID from the bubble grid.
    Returns e.g. '63807' or a string with '?' for unread columns.
    Uses the same robust fixed-grid fill reader as the group grid.
    """
    binary = preprocess(page_gray)
    x, y, w, h = _locate_grid(page_gray, STUDENT_ID_REGION)
    cols = _read_fixed_grid(binary, (x, y, w, h), STUDENT_ID_ROWS, STUDENT_ID_DIGITS)
    return "".join(str(c) if c >= 0 else "?" for c in cols)


def _read_fixed_grid(binary, region, n_rows, n_cols):
    """
    Read a bubble grid by dividing the region into n_rows x n_cols equal cells
    and picking, per column, the row whose box contains an X mark.
    The empty box borders are removed by morphology first, so only the
    hand-drawn cross strokes contribute to the fill measurement.
    Returns a list of length n_cols: the marked row index per column, or -1.
    """
    x0, y0, rw, rh = region
    roi = binary[y0:y0 + rh, x0:x0 + rw]
    inv = cv2.bitwise_not(roi)

    cell_h = rh / n_rows
    cell_w = rw / n_cols

    # Remove straight box borders (long horizontal / vertical runs), keep X strokes
    h_len = max(5, int(cell_w * 0.55))
    v_len = max(5, int(cell_h * 0.55))
    h_kern = cv2.getStructuringElement(cv2.MORPH_RECT, (h_len, 1))
    v_kern = cv2.getStructuringElement(cv2.MORPH_RECT, (1, v_len))
    lines = cv2.add(cv2.morphologyEx(inv, cv2.MORPH_OPEN, h_kern),
                    cv2.morphologyEx(inv, cv2.MORPH_OPEN, v_kern))
    marks = cv2.subtract(inv, lines)

    result = []
    for c in range(n_cols):
        fills = []
        for r in range(n_rows):
            r0, r1 = int(r * cell_h), int((r + 1) * cell_h)
            c0, c1 = int(c * cell_w), int((c + 1) * cell_w)
            cell = marks[r0:r1, c0:c1]
            fills.append(np.sum(cell > 0) / max(cell.size, 1))
        fills = np.array(fills)
        idx = int(np.argmax(fills))
        if fills[idx] > 0.02 and fills[idx] > np.median(fills) * 2.0:
            result.append(idx)
        else:
            result.append(-1)
    return result


def extract_group(page_gray):
    """
    Extract the group code (e.g. '78H') from its bubble grid.
    The grid has 2 digit columns and a separate letter column (A-J), with an
    unequal gap between them, so each part is read from its own region using a
    fixed-grid fill reader (robust to the handwritten header boxes above).
    """
    binary = preprocess(page_gray)

    xd, yd, wd, hd = _locate_grid(page_gray, GROUP_DIGITS_REGION)
    drows = _read_fixed_grid(binary, (xd, yd, wd, hd), GROUP_ROWS, 2)
    digits = "".join(str(r) if r >= 0 else "?" for r in drows)

    xl, yl, wl, hl = _locate_grid(page_gray, GROUP_LETTER_REGION)
    lrows = _read_fixed_grid(binary, (xl, yl, wl, hl), GROUP_ROWS, 1)
    letter = chr(ord('A') + lrows[0]) if lrows[0] >= 0 else "?"

    return digits + letter


def extract_signature_region(page_gray):
    """
    Find the signature box rectangle in the search area and return its interior.
    Falls back to the full search area if no rectangle is found.
    """
    ph, pw = page_gray.shape
    x, y, w, h = _locate_grid(page_gray, SIGNATURE_REGION)
    roi = page_gray[y:y + h, x:x + w]

    # Threshold and find contours of large rectangles
    _, binary = cv2.threshold(roi, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    min_area = (w * h) * 0.05
    max_area = (w * h) * 0.80
    best_area = 0
    best_box  = None

    for cnt in cnts:
        area = cv2.contourArea(cnt)
        if not (min_area < area < max_area):
            continue
        peri = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.04 * peri, True)
        if len(approx) != 4:
            continue
        bx, by, bw, bh = cv2.boundingRect(approx)
        aspect = bw / max(bh, 1)
        if not (0.8 < aspect < 4.0):
            continue
        if area > best_area:
            best_area = area
            best_box  = (bx, by, bw, bh)

    if best_box is not None:
        bx, by, bw, bh = best_box
        pad = 4
        interior = roi[max(0, by + pad): by + bh - pad,
                       max(0, bx + pad): bx + bw - pad]
        if interior.size > 0:
            return interior

    return roi


In [ ]:
%%writefile utils/checkbox_reader.py
"""
Checkbox/bubble detection using morphological operations.
Binarizes the cell, closes small gaps, then checks the dark pixel ratio in the inner area.
"""

import cv2
import numpy as np
from utils.image_processing import preprocess, morpho_close


# ratio of dark pixels above which a checkbox is considered checked
FILL_RATIO_THRESHOLD = 0.15

# margin (pixels) to ignore around the border of a checkbox cell
BORDER_MARGIN = 3


def is_checked(cell_gray):
    """Check whether a checkbox cell is marked. Returns (checked, fill_ratio)."""
    if cell_gray.size == 0:
        return False, 0.0

    binary = preprocess(cell_gray)
    closed = morpho_close(binary, ksize=3)

    # ignore border artifacts
    m = BORDER_MARGIN
    inner = closed[m:-m, m:-m] if min(closed.shape) > 2 * m else closed
    inv = cv2.bitwise_not(inner)

    ratio = float(np.sum(inv > 0)) / inv.size
    return ratio > FILL_RATIO_THRESHOLD, ratio


def read_checkbox_row(page_gray, row_rel_coords, n_choices):
    """Read a row of n_choices checkboxes. Returns list of bool."""
    h, w = page_gray.shape
    x = int(row_rel_coords[0] * w)
    y = int(row_rel_coords[1] * h)
    rw = int(row_rel_coords[2] * w)
    rh = int(row_rel_coords[3] * h)

    roi = page_gray[y:y+rh, x:x+rw]
    cell_w = rw // n_choices

    results = []
    for i in range(n_choices):
        cell = roi[:, i*cell_w:(i+1)*cell_w]
        checked, _ = is_checked(cell)
        results.append(checked)
    return results


In [ ]:
%%writefile utils/cryptogram.py
"""
Cryptogram comparison utilities.
Checks that the small graphic at the bottom of every page matches page 1's cryptogram.
"""

import cv2
import numpy as np
from utils.image_processing import preprocess, normalize_signature, image_similarity
from utils.form_layout import PAGE1_FIELDS, crop_field


CRYPTO_THRESHOLD = 0.70
CRYPTO_SIZE = (64, 32)


def extract_cryptogram(page_gray):
    """Extract and normalize the cryptogram from the bottom of a page."""
    crop = crop_field(page_gray, PAGE1_FIELDS["cryptogram"])
    _, binary = cv2.threshold(crop, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    resized = cv2.resize(binary, CRYPTO_SIZE, interpolation=cv2.INTER_AREA)
    return resized


def validate_cryptograms(pages_gray):
    """Check that all pages share the same cryptogram as page 1. Returns (valid, scores)."""
    if len(pages_gray) == 0:
        return False, []

    ref = extract_cryptogram(pages_gray[0])
    scores = [1.0]

    for page in pages_gray[1:]:
        crypto = extract_cryptogram(page)
        score = image_similarity(ref, crypto)
        scores.append(score)

    valid = all(s >= CRYPTO_THRESHOLD for s in scores[1:])
    return valid, scores


In [ ]:
%%writefile utils/exam_page_parser.py
"""
Exam page parser — block-segmentation approach.

The real form layout (discovered from debug images):
  • Each question sits inside its own full-width bordered rectangle.
  • Multiple-choice questions have their checkboxes stacked VERTICALLY in a
    single column on the left margin (x ≈ 0.08-0.13). Choice A is on top,
    then B, C, D, (E). The marked box is filled with an X / cross.
  • Numerical questions have NO left checkbox column; instead they have
    handwritten boxes in the centre-left: a mantissa box, a "×10" label,
    an exponent box, and a unit box.

Strategy:
  1. Detect question blocks via full-width horizontal border lines (morphology).
  2. For each block:
       - look for checkbox bubbles in the left strip
       - if found  → multiple-choice: the filled bubble's vertical index = choice
       - if absent → numerical: OCR the mantissa / exponent / unit boxes
  3. Return one row per question, in top-to-bottom (page) order.
"""

import cv2
import numpy as np
from utils.image_processing import (preprocess, morpho_open,
                                     find_horizontal_lines)


# ── tunables ──────────────────────────────────────────────────────────────────
MIN_BUBBLE_AREA_RATIO = 0.00015  # min checkbox area as fraction of page area
MAX_BUBBLE_AREA_RATIO = 0.004    # max checkbox area as fraction of page area
ASPECT_RATIO_RANGE    = (0.3, 3.0)   # width/height of a checkbox bounding box

# Left strip where multiple-choice checkboxes live (fraction of page width)
CHECKBOX_X_MIN = 0.04
CHECKBOX_X_MAX = 0.16

# Skip top fraction of each block (contains "● QUESTION N" header band)
BLOCK_HEADER_SKIP = 0.25

# Fill detection
FILL_FLOOR     = 0.14    # absolute dark-pixel ratio above which a box is "marked"
FILL_RELATIVE  = 1.6     # marked box must be ≥ this × the median fill of its group

# Question-block detection
BORDER_MIN_LEN_RATIO = 0.35   # a border line must span ≥ 35 % of the page width
BLOCK_MIN_HEIGHT_RATIO = 0.03 # a question block must be ≥ 3 % of the page height
HEADER_SKIP_RATIO = 0.08      # ignore the top 8 % (page header band)
FOOTER_SKIP_RATIO = 0.04      # ignore the bottom 4 % (page number / cryptogram)

# Numerical answer-box positions (fraction of page width)
# Calibrated from actual PDF: mantissa box x=[0.128:0.251], unit box x=[0.391:0.516]
MANT_X, MANT_W = 0.13, 0.12   # student-written value (x=13%→25%)
EXP_X,  EXP_W  = 0.20, 0.09   # small exponent box above ".10" label (x=20%→29%)
UNIT_X, UNIT_W = 0.40, 0.15   # unit box (x=40%→55%)
# ─────────────────────────────────────────────────────────────────────────────


def _detect_question_blocks(page_gray):
    """
    Return a list of (y0, y1) tuples, one per question block, top-to-bottom.
    Blocks are the tall regions enclosed by full-width horizontal border lines.
    """
    ph, pw = page_gray.shape
    binary = preprocess(page_gray)

    h_mask = find_horizontal_lines(binary, min_len_ratio=BORDER_MIN_LEN_RATIO)
    row_strength = np.sum(h_mask > 0, axis=1)          # how "line-like" each row is
    line_thresh = BORDER_MIN_LEN_RATIO * pw
    is_line = row_strength >= line_thresh

    # Cluster consecutive line rows into single border y-positions
    borders = []
    y = 0
    while y < ph:
        if is_line[y]:
            y_start = y
            while y < ph and is_line[y]:
                y += 1
            borders.append((y_start + y - 1) // 2)
        else:
            y += 1

    # Regions between consecutive borders that are tall enough = question blocks
    min_h = BLOCK_MIN_HEIGHT_RATIO * ph
    y_top_limit = HEADER_SKIP_RATIO * ph
    y_bot_limit = (1 - FOOTER_SKIP_RATIO) * ph

    blocks = []
    for i in range(len(borders) - 1):
        y0, y1 = borders[i], borders[i + 1]
        if (y1 - y0) >= min_h and y0 >= y_top_limit and y1 <= y_bot_limit:
            blocks.append((y0, y1))

    return blocks


def _find_checkboxes_in_strip(page_gray, y0, y1):
    """
    Find checkbox bubbles inside the left strip of the [y0, y1] block.
    Returns list of (cy, fill_ratio) sorted top-to-bottom.
    """
    ph, pw = page_gray.shape
    page_area = ph * pw

    x0 = int(CHECKBOX_X_MIN * pw)
    x1 = int(CHECKBOX_X_MAX * pw)
    # Skip the header band at the top of the block (contains the ● QUESTION N label)
    y_skip = int(y0 + (y1 - y0) * BLOCK_HEADER_SKIP)
    strip = page_gray[y_skip:y1, x0:x1]
    if strip.size == 0:
        return []

    binary = preprocess(strip)
    inv = cv2.bitwise_not(binary)
    opened = morpho_open(inv, ksize=2)

    cnts, _ = cv2.findContours(opened, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

    min_area = MIN_BUBBLE_AREA_RATIO * page_area
    max_area = MAX_BUBBLE_AREA_RATIO * page_area

    dedup_px = 20   # merge contours within 20px (same checkbox, two X strokes)

    seen = {}
    for cnt in cnts:
        bx, by, bw, bh = cv2.boundingRect(cnt)
        area = bw * bh
        if not (min_area < area < max_area):
            continue
        aspect = bw / max(bh, 1)
        if not (ASPECT_RATIO_RANGE[0] < aspect < ASPECT_RATIO_RANGE[1]):
            continue
        pad = max(2, int(min(bw, bh) * 0.18))
        inner = inv[by + pad: by + bh - pad, bx + pad: bx + bw - pad]
        fill = float(np.sum(inner > 0)) / max(inner.size, 1)
        cy = y_skip + by + bh // 2
        key = cy // dedup_px
        if key not in seen or fill > seen[key][1]:
            seen[key] = (cy, fill)

    boxes = sorted(seen.values(), key=lambda b: b[0])
    return boxes


def _clean_binary_for_ocr(binary_black_on_white):
    """
    Remove box borders and small noise from a binary image (black text on white bg).
    Keeps only large connected components (= digit strokes).
    """
    inv = cv2.bitwise_not(binary_black_on_white)  # white text on black
    h = inv.shape[0]
    w = inv.shape[1]
    # Remove long horizontal and vertical lines (box borders)
    h_kern = cv2.getStructuringElement(cv2.MORPH_RECT, (max(15, w // 4), 1))
    v_kern = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(15, h // 4)))
    borders = cv2.add(cv2.morphologyEx(inv, cv2.MORPH_OPEN, h_kern),
                      cv2.morphologyEx(inv, cv2.MORPH_OPEN, v_kern))
    inv_clean = cv2.subtract(inv, borders)
    # Remove small noise components, keep only digit-sized blobs
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(inv_clean, connectivity=8)
    min_area = max(20, inv_clean.size // 600)
    result = np.ones_like(inv_clean) * 255  # white background
    for i in range(1, n_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            result[labels == i] = 0  # black digit
    return result


def _read_number_box(page_gray, x, y, w, h, letters=False):
    """OCR a small handwritten-number box. Returns cleaned string. Skips empty boxes."""
    try:
        import pytesseract, re
    except ImportError:
        return ""
    if w <= 0 or h <= 0:
        return ""
    crop = page_gray[max(0, y):y + h, max(0, x):x + w]
    if crop.size == 0:
        return ""

    # Quick fill check: empty boxes have very few dark pixels after cleaning borders
    if crop.mean() > 250:
        return ""
    # Also skip if dark-pixel ratio is extremely low (border shadows only)
    dark_ratio = float(np.sum(crop < 180)) / max(crop.size, 1)
    if dark_ratio < 0.005:
        return ""

    # Upscale to ~200px height for reliable Tesseract accuracy
    target_h = 200
    scale = max(2, target_h // max(crop.shape[0], 1))
    crop_up = cv2.resize(crop, (crop.shape[1] * scale, crop.shape[0] * scale),
                         interpolation=cv2.INTER_CUBIC)

    # CLAHE to boost faint handwriting contrast, then adaptive MEAN C=2
    clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(4, 4))
    crop_enh = clahe.apply(crop_up)
    block_up = max(11, (min(crop_up.shape) // 4) | 1)
    binary = cv2.adaptiveThreshold(
        crop_enh, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, block_up, 2)

    # Remove box borders and noise → clean binary with black text on white
    binary_clean = _clean_binary_for_ocr(binary)

    whitelist = ("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
                 if letters else "0123456789.-")
    text = pytesseract.image_to_string(
        binary_clean, config=f"--psm 7 -c tessedit_char_whitelist={whitelist}").strip()
    if not text:
        text = pytesseract.image_to_string(binary_clean, config="--psm 6").strip()
        text = re.sub(r"[^0-9A-Za-z.\-]", "", text)
    return text


def parse_exam_page(page_gray, choice_labels=None, page_idx=0, debug_dir=None):
    """
    Parse one exam page using block segmentation.
    Returns list of dicts: [{QUESTION, CHOIX, MANTISSE, EXPOSANT, UNITE}, ...]
    """
    if choice_labels is None:
        choice_labels = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']

    ph, pw = page_gray.shape
    blocks = _detect_question_blocks(page_gray)

    debug_items = []   # (y0, y1, checkboxes, marked_idx, kind)
    rows = []

    for (y0, y1) in blocks:
        checkboxes = _find_checkboxes_in_strip(page_gray, y0, y1)

        row = {"QUESTION": 0, "CHOIX": "", "MANTISSE": "", "EXPOSANT": "", "UNITE": ""}

        if len(checkboxes) >= 3:
            # ── Multiple-choice question ──
            fills = np.array([f for (_, f) in checkboxes])
            best = int(np.argmax(fills))
            med = float(np.median(fills))
            marked = None
            if fills[best] >= FILL_FLOOR and fills[best] >= med * FILL_RELATIVE:
                marked = best
                if best < len(choice_labels):
                    row["CHOIX"] = choice_labels[best]
                else:
                    row["CHOIX"] = str(best)
            debug_items.append((y0, y1, checkboxes, marked, "mcq"))
        else:
            # ── Numerical question ──
            # Answer boxes sit in the very bottom of the block (~last 25%)
            box_h  = max(30, int((y1 - y0) * 0.25))
            by_num = max(0, y1 - box_h - 3)
            # Exponent box is printed ABOVE the ".10" label, roughly one box-height
            # above the mantissa row
            exp_h  = max(20, int((y1 - y0) * 0.14))
            by_exp = max(0, by_num - exp_h - 4)
            row["MANTISSE"] = _read_number_box(page_gray, int(MANT_X * pw), by_num,
                                               int(MANT_W * pw), box_h)
            row["EXPOSANT"] = _read_number_box(page_gray, int(EXP_X * pw), by_exp,
                                               int(EXP_W * pw), exp_h)
            row["UNITE"]    = _read_number_box(page_gray, int(UNIT_X * pw), by_num,
                                               int(UNIT_W * pw), box_h, letters=True)
            debug_items.append((y0, y1, checkboxes, None, "num"))

        rows.append(row)

    # Number questions 1..N in page order
    for i, row in enumerate(rows):
        row["QUESTION"] = i + 1

    if debug_dir is not None:
        _save_debug(page_gray, debug_items, debug_dir, page_idx)

    return rows


def _save_debug(page_gray, debug_items, debug_dir, page_idx):
    """Save an annotated image showing detected blocks, checkboxes and marks."""
    import os
    os.makedirs(debug_dir, exist_ok=True)
    ph, pw = page_gray.shape
    vis = cv2.cvtColor(page_gray, cv2.COLOR_GRAY2BGR)

    x0 = int(CHECKBOX_X_MIN * pw)
    x1 = int(CHECKBOX_X_MAX * pw)

    for (y0, y1, checkboxes, marked, kind) in debug_items:
        # block outline
        bcol = (255, 150, 0) if kind == "mcq" else (200, 0, 200)
        cv2.rectangle(vis, (2, y0), (pw - 3, y1), bcol, 1)
        # checkboxes
        for idx, (cy, fill) in enumerate(checkboxes):
            col = (0, 0, 255) if idx == marked else (0, 200, 0)
            cv2.rectangle(vis, (x0, cy - 10), (x1, cy + 10), col, 2)
            cv2.putText(vis, f"{fill:.2f}", (x1 + 4, cy + 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, col, 1)
        if kind == "num":
            cv2.putText(vis, "NUM", (x0, y0 + 18),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 0, 200), 2)
            # Draw the numerical answer box region
            box_h  = max(30, int((y1 - y0) * 0.25))
            by_num = max(0, y1 - box_h - 3)
            cv2.rectangle(vis,
                          (int(MANT_X * pw), by_num),
                          (int((MANT_X + MANT_W) * pw), by_num + box_h),
                          (0, 200, 255), 2)

    out = os.path.join(debug_dir, f"page_{page_idx:02d}_blocks.png")
    cv2.imwrite(out, vis)
    # Also save individual num-box crops for calibration
    num_q = 0
    for (y0, y1, checkboxes, marked, kind) in debug_items:
        if kind == "num":
            box_h  = max(30, int((y1 - y0) * 0.25))
            by_num = max(0, y1 - box_h - 3)
            mx = int(MANT_X * pw)
            mw = int(MANT_W * pw)
            crop = page_gray[by_num:by_num + box_h, mx:mx + mw]
            cv2.imwrite(os.path.join(debug_dir,
                        f"page_{page_idx:02d}_numQ{num_q}_mantissa.png"), crop)
            num_q += 1


In [ ]:
%%writefile utils/signature_matcher.py
"""
Matches a signature against the class database.
Uses a weighted combination of HOG, Hu Moments, and projection profiles —
more robust to geometric distortion than NCC (which fails on camera photos).
"""

from pathlib import Path
import cv2
import numpy as np
from utils.image_processing import load_image, normalize_signature


SIG_TARGET_SIZE = (128, 64)   # (width, height)
# 1:N identification accept threshold. Set near the validated 1:1 verification
# threshold (0.64): in identification the score is a max over all students, so
# the impostor competition is stronger and a permissive value (e.g. 0.30) would
# accept almost anything. 0.60 keeps only confident matches.
MATCH_THRESHOLD = 0.60        # combined score threshold

SUPPORTED_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif",
                 ".heic", ".heif", ".webp"}


def _sig_score_hog(a, b):
    def hog_hist(img):
        gx = cv2.Sobel(img.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(img.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)
        mag, ang = cv2.cartToPolar(gx, gy, angleInDegrees=True)
        hist, _ = np.histogram(ang[mag > 10], bins=8, range=(0, 360),
                               weights=mag[mag > 10])
        norm = np.linalg.norm(hist)
        return hist / norm if norm > 0 else hist
    return float(np.dot(hog_hist(a), hog_hist(b)))


def _sig_score_moments(a, b):
    def hu(img):
        m = cv2.moments(img)
        h = cv2.HuMoments(m).flatten()
        return -np.sign(h) * np.log10(np.abs(h) + 1e-10)
    return 1.0 / (1.0 + np.sum(np.abs(hu(a) - hu(b))))


def _sig_score_projection(a, b):
    def profiles(img):
        ph = np.sum(img > 0, axis=1).astype(np.float32)
        pv = np.sum(img > 0, axis=0).astype(np.float32)
        ph /= ph.max() + 1e-6
        pv /= pv.max() + 1e-6
        return np.concatenate([ph, pv])
    corr = np.corrcoef(profiles(a), profiles(b))[0, 1]
    return float(max(0.0, corr))


def _compare_signatures(a, b):
    """Weighted combination: HOG 50% + Projection 30% + Moments 20%."""
    return (0.5 * _sig_score_hog(a, b)
            + 0.3 * _sig_score_projection(a, b)
            + 0.2 * _sig_score_moments(a, b))


def _load_database(signatures_dir):
    db = {}

    def _add(path, student_id):
        try:
            _, img = load_image(path)
        except Exception:
            return
        norm = normalize_signature(img, SIG_TARGET_SIZE)
        if np.sum(norm) > 0:
            db.setdefault(student_id, []).append(norm)

    def _walk(folder, current_id):
        for item in sorted(Path(folder).iterdir()):
            if item.is_dir():
                sid = item.name if item.name.isdigit() else current_id
                _walk(item, sid)
            elif item.suffix.lower() in SUPPORTED_EXT:
                sid = current_id or item.stem
                if sid:
                    _add(item, sid)

    _walk(signatures_dir, None)
    return db


_db_cache = {}


def match_signature(sig_gray, signatures_dir):
    """
    Identify a signature. Returns (best_id, best_score).
    best_id is None if score < MATCH_THRESHOLD.
    """
    key = str(signatures_dir)
    if key not in _db_cache:
        _db_cache[key] = _load_database(signatures_dir)
    db = _db_cache[key]

    if not db:
        return None, 0.0

    query = normalize_signature(sig_gray, SIG_TARGET_SIZE)
    if np.sum(query) == 0:
        return None, 0.0

    best_id, best_score = None, -1.0
    for student_id, refs in db.items():
        score = max(_compare_signatures(query, ref) for ref in refs)
        if score > best_score:
            best_score = score
            best_id = student_id

    if best_score < MATCH_THRESHOLD:
        return None, best_score
    return best_id, best_score


# Default operating threshold, optimised on the validation set via the strict
# train/validation/test protocol in optimize_threshold.py (balanced accuracy of
# genuine-accept vs impostor-reject). Train & validation optima both gave 0.64.
VERIFY_THRESHOLD = 0.64


def signature_score(sig_gray, student_id, signatures_dir):
    """
    Raw similarity score between a query signature and the references of
    student_id (max over that student's reference signatures).
    Returns 0.0 if the student is unknown or the query is empty.
    Used by the evaluation harness to sweep the decision threshold.
    """
    key = str(signatures_dir)
    if key not in _db_cache:
        _db_cache[key] = _load_database(signatures_dir)
    db = _db_cache[key]

    refs = db.get(str(student_id), [])
    if not refs:
        return 0.0

    query = normalize_signature(sig_gray, SIG_TARGET_SIZE)
    if np.sum(query) == 0:
        return 0.0

    return max(_compare_signatures(query, ref) for ref in refs)


def verify_signature(sig_gray, student_id, signatures_dir, threshold=None):
    """
    Verify that sig_gray belongs to student_id (1:1 authentication).
    The decision threshold defaults to VERIFY_THRESHOLD but can be overridden
    (e.g. with a value optimised on a validation set).
    Returns (matched: bool, score: float).
    """
    if threshold is None:
        threshold = VERIFY_THRESHOLD
    score = signature_score(sig_gray, student_id, signatures_dir)
    return score >= threshold, score


In [ ]:
%%writefile utils/ground_truth.py
"""
Ground-truth extraction from file names.

The provided data encodes the true student ID in each file name, e.g.
    EXAM_FORM1_63807.jpeg   -> student 63807   (presence photo)
    EXAM_FORM1_0001.pdf     -> (form index, not a student ID)

This lets us build an annotated evaluation set automatically, as required by
the methodology (Section 4.2): a labelled base for train / validation / test.
"""

import re
from pathlib import Path

# A student ID is a 5-digit number; form indices are 4-digit (0001..0068).
_STUDENT_ID_RE = re.compile(r"_(\d{5})(?:\b|_|\.)")


def student_id_from_name(filename):
    """
    Return the ground-truth student ID embedded in a file name, or None.
    e.g. 'EXAM_FORM1_63807.jpeg' -> '63807'
    """
    stem = Path(filename).stem
    m = _STUDENT_ID_RE.search(stem)
    return m.group(1) if m else None


def list_labelled_images(presences_dir, extensions=None):
    """
    Yield (path, ground_truth_id) for every labelled image in a presences dir.
    Skips files whose name does not encode a 5-digit student ID.
    """
    if extensions is None:
        extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif",
                      ".heic", ".heif", ".webp"}
    pairs = []
    for path in sorted(Path(presences_dir).iterdir()):
        if path.suffix.lower() not in extensions:
            continue
        gt = student_id_from_name(path.name)
        if gt is not None:
            pairs.append((path, gt))
    return pairs


In [ ]:
%%writefile autoValidPresences.py
"""Validation des présences — lit le formulaire, extrait l'ID et la signature."""

from pathlib import Path
import cv2
import numpy as np
import openpyxl

from utils.image_processing import load_image, deskew, correct_perspective
from utils.grid_reader import extract_student_id, extract_signature_region
from utils.signature_matcher import match_signature, verify_signature


# All image extensions that may appear in the presences folder
SUPPORTED_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif",
                 ".heic", ".heif", ".webp", ".JPG", ".PNG", ".JPEG"}


def _prepare_image(img_gray):
    """
    Prepare a presence photo for processing:
    1. Resize to a standard height (avoids memory issues with 12 MP photos)
    2. Deskew
    """
    # Cap height at 2000 px to standardise resolution
    h, w = img_gray.shape
    if h > 2000:
        scale = 2000 / h
        img_gray = cv2.resize(img_gray, (int(w * scale), 2000),
                              interpolation=cv2.INTER_AREA)

    img_gray, _ = deskew(img_gray)

    # NB: an L-bracket perspective correction was explored (see
    # utils.image_processing.correct_perspective) but, on these fairly frontal
    # camera photos, the deskew already removes most distortion and the warp did
    # not improve grid accuracy on the validation set, so it is left disabled.
    return img_gray


def autoValidID(image_path, signatures_dir, xlsx_path, results_dir):
    """
    Process one presence photo.
    Returns (image_name, student_id_grid, student_id_signature).
    student_id_signature is None if no match found.
    """
    image_path = Path(image_path)
    _, img_gray = load_image(image_path)
    img_gray = _prepare_image(img_gray)

    student_id_grid = extract_student_id(img_gray)
    sig_gray = extract_signature_region(img_gray)

    # Column C (studentID_signature = StudentID_bitmap, cf. §3.3): the identity
    # the signature itself belongs to, deduced from the signature database —
    # INDEPENDENT of the grid, so the professor can compare B and C to spot an
    # identity usurpation (B != C) or an unrecognised signature (C empty).
    #
    # We first test the fast, accurate 1:1 hypothesis "the signature confirms the
    # grid ID"; if it is not confirmed (wrong/usurped/unreadable grid), we fall
    # back to a full 1:N identification so column C is still populated.
    verified, score = verify_signature(sig_gray, student_id_grid, signatures_dir)
    if verified and "?" not in student_id_grid:
        student_id_sig = student_id_grid
    else:
        student_id_sig, score = match_signature(sig_gray, signatures_dir)

    status = "✓" if (student_id_sig and student_id_sig == student_id_grid) else "?"
    print(f"  {status} [{image_path.name}]  "
          f"grid={student_id_grid}  sig={student_id_sig}  (score={score:.3f})")

    return image_path.name, student_id_grid, student_id_sig


def autoValidPresences(exam_presences_dir, signatures_dir, results_dir):
    """
    Process all presence photos in exam_presences_dir.
    Writes EXAM_FORMXX_PRESENCES.xlsx with columns:
        imageName | studentID_grid | studentID_signature
    """
    exam_presences_dir = Path(exam_presences_dir)
    signatures_dir     = Path(signatures_dir)
    results_dir        = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)

    exam_name = exam_presences_dir.name.replace("_PRESENCES", "")
    xlsx_path = results_dir / f"{exam_name}_PRESENCES.xlsx"

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "PRESENCES"
    ws.append(["imageName", "studentID_grid", "studentID_signature"])

    # Collect all image files (case-insensitive extension match)
    images = sorted([
        f for f in exam_presences_dir.iterdir()
        if f.suffix.lower() in {e.lower() for e in SUPPORTED_EXT}
    ])

    if not images:
        print(f"[WARNING] No images found in {exam_presences_dir}")

    ok = wrong = unmatched = 0
    for img_path in images:
        try:
            name, id_grid, id_sig = autoValidID(
                img_path, signatures_dir, xlsx_path, results_dir)
            ws.append([name, id_grid, id_sig if id_sig else ""])
            if id_sig and id_sig == id_grid:
                ok += 1
            elif id_sig is None:
                unmatched += 1
            else:
                wrong += 1
        except Exception as exc:
            print(f"  [ERROR] {img_path.name}: {exc}")
            ws.append([img_path.name, "", ""])

    wb.save(str(xlsx_path))
    print(f"\n[Programme 1] {len(images)} images — "
          f"{ok} OK / {unmatched} non reconnues / {wrong} suspectes")
    print(f"  → {xlsx_path}")
    return str(xlsx_path)


In [ ]:
%%writefile autoReadForm.py
"""Lecture automatique des formulaires d'examen — Programme 2."""

import os
import re
from pathlib import Path

import cv2
import numpy as np
import openpyxl

from utils.pdf_utils import pdf_to_images
from utils.image_processing import deskew, preprocess
from utils.grid_reader import (extract_student_id, extract_group,
                                extract_signature_region)
from utils.signature_matcher import match_signature, verify_signature
from utils.cryptogram import validate_cryptograms, extract_cryptogram
from utils.ocr_reader import read_printed_field, read_printed_date
from utils.checkbox_reader import is_checked
from utils.form_layout import PAGE1_FIELDS, crop_field
from utils.exam_page_parser import parse_exam_page


# Choices available per question — up to 8 (A-H) as per spec
CHOICE_LABELS = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']


def _strip_ocr_noise(text):
    """Remove trailing punctuation and whitespace from OCR output."""
    return re.sub(r'[\s|:.,;]+$', '', text.strip())


def _fix_module(text):
    """Fix OCR errors in module code (format LL.DDDD, e.g. IG.1103)."""
    text = _strip_ocr_noise(text)
    d2l = {'1': 'I', '6': 'G', '0': 'O', '5': 'S', '8': 'B', '3': 'E', '4': 'A'}
    if len(text) >= 2:
        chars = list(text)
        for i in (0, 1):
            if chars[i].isdigit():
                chars[i] = d2l.get(chars[i], chars[i])
        # Fix separator: position 2 should be '.'
        if len(chars) >= 3 and chars[2] in ('-', ',', ';', '_', ' '):
            chars[2] = '.'
        text = ''.join(chars)
    return text


def _fix_code(text):
    """Fix OCR errors in exam code (format LN-NN-LN, e.g. S1-01-G1)."""
    text = _strip_ocr_noise(text)
    d2l = {'3': 'S', '5': 'S', '6': 'G', '0': 'O', '1': 'I', '8': 'B'}
    if len(text) >= 1 and text[0].isdigit():
        text = d2l.get(text[0], text[0]) + text[1:]
    if len(text) >= 7 and text[6].isdigit():
        text = text[:6] + d2l.get(text[6], text[6]) + text[7:]
    return text


def _read_field_ocr(page_gray, rel_coords, mode="printed"):
    """Crop a field and OCR it with CLAHE + adaptive threshold for robust reading."""
    crop = crop_field(page_gray, rel_coords)
    if crop.size == 0:
        return ""

    # Upscale to at least 60px height for reliable OCR
    h, w = crop.shape
    scale = max(1, 60 // max(h, 1))
    if scale > 1:
        crop = cv2.resize(crop, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)

    try:
        import pytesseract
    except ImportError:
        return ""

    # CLAHE to handle gray-shaded boxes, then adaptive threshold
    clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(4, 4))
    crop_enh = clahe.apply(crop)
    block = max(11, (min(crop_enh.shape) // 3) | 1)
    binary = cv2.adaptiveThreshold(crop_enh, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                    cv2.THRESH_BINARY, block, 3)

    if mode == "date":
        cfg = "--psm 7 -c tessedit_char_whitelist=0123456789/"
        t = pytesseract.image_to_string(binary, config=cfg).strip()
        if not t:
            _, bin_otsu = cv2.threshold(crop, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            t = pytesseract.image_to_string(bin_otsu, config=cfg).strip()
        return t

    if mode == "digits":
        # Gray halftone-stippled boxes with gray printed digits. Median blur
        # removes the dot screen, Otsu then separates the gray digits from the
        # lighter background. Vote across PSM/scale, preferring the longest read.
        from collections import Counter
        ch, cw = crop.shape
        my, mx = int(ch * 0.10), int(cw * 0.05)
        inner = crop[my:ch - my, mx:cw - mx] if ch > 2 * my and cw > 2 * mx else crop
        cfgs = ["--psm 8 -c tessedit_char_whitelist=0123456789",
                "--psm 7 -c tessedit_char_whitelist=0123456789"]
        candidates = []
        for ksize in (5, 3):
            med = cv2.medianBlur(inner, ksize)
            up = cv2.resize(med, (med.shape[1] * 3, med.shape[0] * 3),
                            interpolation=cv2.INTER_CUBIC)
            _, b = cv2.threshold(up, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            for cfg in cfgs:
                t = re.sub(r'[^0-9]', '', pytesseract.image_to_string(b, config=cfg).strip())
                if t:
                    candidates.append(t)
        if not candidates:
            return ""
        counts = Counter(candidates)
        return max(counts, key=lambda k: (counts[k], len(k)))

    if mode == "handwriting":
        # Remove vertical box borders then OCR the remaining letter strokes
        inv = cv2.bitwise_not(crop_enh)
        v_kern = cv2.getStructuringElement(cv2.MORPH_RECT,
                                           (1, max(10, crop_enh.shape[0] // 3)))
        vlines = cv2.morphologyEx(inv, cv2.MORPH_OPEN, v_kern)
        inv_clean = cv2.subtract(inv, vlines)
        h_kern = cv2.getStructuringElement(cv2.MORPH_RECT,
                                           (max(10, crop_enh.shape[1] // 6), 1))
        hlines = cv2.morphologyEx(inv_clean, cv2.MORPH_OPEN, h_kern)
        inv_clean = cv2.subtract(inv_clean, hlines)
        cleaned = cv2.bitwise_not(inv_clean)
        t = pytesseract.image_to_string(cleaned, config="--psm 6").strip()
        if not t:
            t = pytesseract.image_to_string(binary, config="--psm 6").strip()
        # Remove box-artifact characters
        t = re.sub(r'[|_\[\]{}]', '', t).strip()
        return t

    t = pytesseract.image_to_string(binary, config="--psm 7").strip()
    if not t:
        _, bin_otsu = cv2.threshold(crop, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        t = pytesseract.image_to_string(bin_otsu, config="--psm 7").strip()
    return t


def _checkbox_val(page_gray, rel_coords):
    """Check whether the checkbox at rel_coords is marked."""
    cell = crop_field(page_gray, rel_coords)
    if cell.size == 0:
        return 0
    checked, ratio = is_checked(cell)
    return 1 if checked else 0


def _parse_page1(page_gray, signatures_dir):
    """Extract all PAGE-01 fields and return them as a dict."""
    page_gray, _ = deskew(page_gray)

    data = {}

    data["Module"]    = _fix_module(_read_field_ocr(page_gray, PAGE1_FIELDS["module"]))
    data["Professor"] = _strip_ocr_noise(_read_field_ocr(page_gray, PAGE1_FIELDS["professor"]))
    data["Date"]      = _read_field_ocr(page_gray, PAGE1_FIELDS["date"], mode="date")
    data["Code"]      = _fix_code(_read_field_ocr(page_gray, PAGE1_FIELDS["code"]))

    data["Notes de cours"]      = _checkbox_val(page_gray, PAGE1_FIELDS["notes_cours"])
    data["Notes manuscrites"]   = _checkbox_val(page_gray, PAGE1_FIELDS["notes_manuscrites"])
    data["Ordinateur portable"] = _checkbox_val(page_gray, PAGE1_FIELDS["ordinateur"])
    data["Calculatrice"]        = _checkbox_val(page_gray, PAGE1_FIELDS["calculatrice"])
    data["Feuilles brouillon"]  = _checkbox_val(page_gray, PAGE1_FIELDS["feuilles_brouillon"])

    data["Note maximale"]     = _read_field_ocr(page_gray, PAGE1_FIELDS["note_maximale"], mode="digits")
    data["Note pour valider"] = _read_field_ocr(page_gray, PAGE1_FIELDS["note_valider"], mode="digits")

    data["Prénom"] = _read_field_ocr(page_gray, PAGE1_FIELDS["prenom"], mode="handwriting")
    data["Nom"]    = _read_field_ocr(page_gray, PAGE1_FIELDS["nom"],    mode="handwriting")

    data["Group"]      = extract_group(page_gray)
    data["STUDENT ID"] = extract_student_id(page_gray)

    # Signature authentication: verify (1:1) against the grid-read student ID,
    # which is reliable on scanned PDFs. Fall back to 1:N if the grid is unreadable.
    sig_gray = extract_signature_region(page_gray)
    student_id = data["STUDENT ID"]
    if student_id and "?" not in student_id:
        verified, sig_score = verify_signature(sig_gray, student_id, signatures_dir)
        student_id_sig = student_id if verified else None
    else:
        student_id_sig, sig_score = match_signature(sig_gray, signatures_dir)
    data["Validation signature"] = 1 if student_id_sig else 0
    data["_signature_id"]    = student_id_sig or ""
    data["_signature_score"] = sig_score

    return data


def _parse_exam_pages(pages_gray, debug_dir=None):
    """
    Parse exam answer pages (pages 3-7, index 2 onward, skipping page 2 which is blank).
    Returns a list of question-row dicts with sequential question numbers.
    """
    exam_rows = []
    q_offset = 0

    # Spec: "pages d'examens (p5→fin)" → index 4 onward (pages 1-4 = identity + preamble)
    for page_idx, page in enumerate(pages_gray[4:], start=4):
        page_gray, _ = deskew(page)
        rows = parse_exam_page(page_gray, CHOICE_LABELS,
                               page_idx=page_idx, debug_dir=debug_dir)
        for row in rows:
            row["QUESTION"] = q_offset + row["QUESTION"]
            exam_rows.append(row)
        q_offset += len(rows)

    return exam_rows


def _build_xlsx(page1_data, exam_rows, crypto_valid, xlsx_path):
    """Write the output XLSX with PAGE-01 and EXAM sheets."""
    wb = openpyxl.Workbook()

    ws1 = wb.active
    ws1.title = "PAGE-01"
    ws1.append(["Field", "Value"])
    for label, key in [
        ("Module",               "Module"),
        ("Professor",            "Professor"),
        ("Date",                 "Date"),
        ("Code",                 "Code"),
        ("Notes de cours",       "Notes de cours"),
        ("Notes manuscrites",    "Notes manuscrites"),
        ("Ordinateur portable",  "Ordinateur portable"),
        ("Calculatrice",         "Calculatrice"),
        ("Feuilles brouillon",   "Feuilles brouillon"),
        ("Note maximale",        "Note maximale"),
        ("Note pour valider",    "Note pour valider"),
        ("",                     ""),
        ("Prénom",               "Prénom"),
        ("Nom",                  "Nom"),
        ("Validation signature", "Validation signature"),
        ("Group",                "Group"),
        ("STUDENT ID",           "STUDENT ID"),
        ("Validation cryptogramme", "Validation cryptogramme"),
    ]:
        ws1.append([label, page1_data.get(key, "")])

    ws2 = wb.create_sheet(title="EXAM")
    if exam_rows:
        # Build headers: QUESTION, CHOIX A, CHOIX B, …, CHOIX H, MANTISSE, EXPOSANT, UNITE
        choice_headers = [f"CHOIX {c}" for c in CHOICE_LABELS]
        headers = ["QUESTION"] + choice_headers + ["MANTISSE", "EXPOSANT", "UNITE"]
        ws2.append(headers)
        for row in exam_rows:
            chosen = row.get("CHOIX", "")
            values = [row.get("QUESTION", "")]
            for c in CHOICE_LABELS:
                values.append(1 if chosen == c else 0)
            values.append(row.get("MANTISSE", ""))
            values.append(row.get("EXPOSANT", ""))
            values.append(row.get("UNITE", ""))
            ws2.append(values)

    wb.save(str(xlsx_path))


def autoReadFormID(pdf_path, signatures_dir, results_dir, debug=False):
    """Read one exam PDF and write the corresponding XLSX."""
    pdf_path = Path(pdf_path)
    signatures_dir = Path(signatures_dir)
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)

    xlsx_path = results_dir / (pdf_path.stem + ".xlsx")
    debug_dir = str(results_dir / "debug") if debug else None

    print(f"  Processing {pdf_path.name} …")

    pages = pdf_to_images(pdf_path)
    if not pages:
        print(f"  [ERROR] No pages found in {pdf_path.name}")
        return

    page1_data = _parse_page1(pages[0], signatures_dir)

    crypto_valid, crypto_scores = validate_cryptograms(pages)
    page1_data["Validation cryptogramme"] = 1 if crypto_valid else 0
    print(f"    cryptogram valid={crypto_valid}  scores={[f'{s:.2f}' for s in crypto_scores]}")

    exam_rows = _parse_exam_pages(pages, debug_dir=debug_dir)

    _build_xlsx(page1_data, exam_rows, crypto_valid, xlsx_path)

    print(f"    → {xlsx_path.name}  "
          f"(studentID={page1_data.get('STUDENT ID', '')}  "
          f"sig={page1_data.get('_signature_id', '')}  "
          f"questions={len(exam_rows)})")

    return str(xlsx_path)


def autoReadForm(exam_pdf_dir, signatures_dir, results_dir, debug=False):
    """Process all PDFs in exam_pdf_dir."""
    exam_pdf_dir = Path(exam_pdf_dir)
    signatures_dir = Path(signatures_dir)
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)

    pdfs = sorted(exam_pdf_dir.glob("*.pdf"))
    if not pdfs:
        print(f"[WARNING] No PDF files found in {exam_pdf_dir}")
        return

    print(f"[Programme 2] Found {len(pdfs)} PDF(s) in {exam_pdf_dir}")
    for pdf in pdfs:
        try:
            autoReadFormID(pdf, signatures_dir, results_dir, debug=debug)
        except Exception as exc:
            import traceback
            print(f"  [ERROR] {pdf.name}: {exc}")
            traceback.print_exc()

    print(f"\n[Programme 2] All results saved in {results_dir}")


In [ ]:
%%writefile optimize_threshold.py
"""
Signature-threshold optimisation with a strict train / validation / test
protocol (methodology — Section 4.2).

The signature decision threshold is:
  • OPTIMISED on the training set,
  • SELECTED (cross-checked) on the validation set,
  • and the final performance is REPORTED on the held-out test set,
which is never used for tuning.

Ground truth is read automatically from file names (utils.ground_truth), so no
manual annotation is needed. For the signature axis we build:
  • genuine pairs  : (photo signature, its true ID)        → should ACCEPT
  • impostor pairs : (photo signature, another student ID) → should REJECT
and optimise the threshold for balanced accuracy — exactly the trade-off the
challenge probes by injecting identity-usurpation cases.

Usage:
    python optimize_threshold.py TRAIN_DIR VAL_DIR TEST_DIR \
           --signatures STUDENT_CLASS_SIGNATURES
e.g.
    python optimize_threshold.py /content/EXAM_FORM1_PRESENCES \
        /content/EXAM_FORM2_PRESENCES /content/EXAM_FORM3_PRESENCES \
        --signatures /content/STUDENT_CLASS_SIGNATURES
"""

import argparse
import random

import cv2
import numpy as np

from utils.image_processing import load_image, deskew, correct_perspective
from utils.grid_reader import extract_student_id, extract_signature_region
from utils.signature_matcher import signature_score
from utils.ground_truth import list_labelled_images


def _prepare(img_gray):
    """Standardise resolution and deskew, mirroring autoValidPresences."""
    h, w = img_gray.shape
    if h > 2000:
        scale = 2000 / h
        img_gray = cv2.resize(img_gray, (int(w * scale), 2000),
                              interpolation=cv2.INTER_AREA)
    img_gray, _ = deskew(img_gray)
    return img_gray


def collect_scores(presences_dir, signatures_dir, seed=0):
    """
    Run the pipeline once over a presences directory and collect, per image:
      • grid_correct   : was the bubble-grid student ID read correctly?
      • genuine_score  : signature score against the true ID
      • impostor_score : signature score against a random different ID
    """
    rng = random.Random(seed)
    pairs = list_labelled_images(presences_dir)
    all_ids = [gt for _, gt in pairs]

    records = []
    for path, gt in pairs:
        try:
            _, gray = load_image(path)
        except Exception:
            continue
        gray = _prepare(gray)

        grid_id = extract_student_id(gray)
        sig = extract_signature_region(gray)

        genuine = signature_score(sig, gt, signatures_dir)
        others = [i for i in all_ids if i != gt]
        impostor_id = rng.choice(others) if others else gt
        impostor = signature_score(sig, impostor_id, signatures_dir)

        records.append({
            "name": path.name, "gt": gt, "grid_id": grid_id,
            "grid_correct": grid_id == gt,
            "genuine_score": genuine, "impostor_score": impostor,
        })
    return records


def grid_accuracy(records):
    if not records:
        return 0.0
    return sum(r["grid_correct"] for r in records) / len(records)


def balanced_accuracy(records, threshold):
    """0.5 * (genuine-accept rate + impostor-reject rate) at a threshold."""
    if not records:
        return 0.0
    gen = np.array([r["genuine_score"] for r in records])
    imp = np.array([r["impostor_score"] for r in records])
    tpr = np.mean(gen >= threshold)
    tnr = np.mean(imp < threshold)
    return 0.5 * (tpr + tnr)


def optimise_threshold(records, grid=None):
    if grid is None:
        grid = np.round(np.arange(0.30, 0.80, 0.01), 3)
    best_t, best_acc = grid[0], -1.0
    for t in grid:
        acc = balanced_accuracy(records, t)
        if acc > best_acc:
            best_acc, best_t = acc, t
    return best_t, best_acc


def _report(name, records, threshold):
    gen = np.array([r["genuine_score"] for r in records])
    imp = np.array([r["impostor_score"] for r in records])
    print(f"\n── {name}  ({len(records)} images) ──")
    print(f"  StudentID grid accuracy   : {grid_accuracy(records):6.1%}")
    print(f"  Signature balanced acc.   : {balanced_accuracy(records, threshold):6.1%}"
          f"  (threshold={threshold:.2f})")
    print(f"    genuine  score mean={gen.mean():.3f} median={np.median(gen):.3f}")
    print(f"    impostor score mean={imp.mean():.3f} median={np.median(imp):.3f}")
    print(f"    genuine accept rate (TPR) : {np.mean(gen >= threshold):.1%}")
    print(f"    impostor reject rate (TNR): {np.mean(imp <  threshold):.1%}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("train"); ap.add_argument("val"); ap.add_argument("test")
    ap.add_argument("--signatures", required=True)
    args = ap.parse_args()

    print("Collecting scores (train/val/test) …")
    train = collect_scores(args.train, args.signatures)
    val   = collect_scores(args.val,   args.signatures)
    test  = collect_scores(args.test,  args.signatures)

    t_train, acc_train = optimise_threshold(train)
    print(f"\n[TRAIN] optimal threshold = {t_train:.2f} (balanced acc {acc_train:.1%})")

    t_val, acc_val = optimise_threshold(val)
    print(f"[VALIDATION] own optimum = {t_val:.2f} ({acc_val:.1%}); "
          f"train threshold scores {balanced_accuracy(val, t_train):.1%} here")

    operating_t = round((t_train + t_val) / 2, 2)
    print(f"\n>>> Selected operating threshold = {operating_t:.2f}")
    print(f"    (set utils/signature_matcher.VERIFY_THRESHOLD = {operating_t:.2f})")

    for name, recs in [("TRAIN", train), ("VALIDATION", val), ("TEST", test)]:
        _report(name, recs, operating_t)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate.py
"""
Évaluation quantitative des programmes autoValidPresences et autoReadForm.
Compare les sorties XLSX avec les vérités terrain.

Usage (Colab):
    !python evaluate.py /content/EXAM_FORM1_RESULTS /content/drive/.../GT_FORM1

Structure attendue :
    results_dir/
        EXAM_FORM1_PRESENCES.xlsx      ← sortie Programme 1
        EXAM_FORM1_63807.xlsx          ← sortie Programme 2 (un par PDF)
    ground_truth_dir/
        EXAM_FORM1_PRESENCES.xlsx      ← vérité terrain Programme 1
        EXAM_FORM1_63807.xlsx          ← vérité terrain Programme 2
"""

import sys
from pathlib import Path
import openpyxl


# ── helpers ──────────────────────────────────────────────────────────────────

def _load_sheet(xlsx_path, sheet=0):
    wb = openpyxl.load_workbook(str(xlsx_path), read_only=True, data_only=True)
    ws = wb.worksheets[sheet] if isinstance(sheet, int) else wb[sheet]
    rows = list(ws.iter_rows(values_only=True))
    if not rows:
        return []
    headers = [str(h).strip() if h is not None else "" for h in rows[0]]
    return [{headers[i]: row[i] for i in range(len(headers))} for row in rows[1:]]


def _norm(v):
    """Normalize cell value: strip, upper, remove spaces, cast numeric."""
    s = str(v).strip().upper().replace(" ", "") if v is not None else ""
    try:
        return str(int(float(s)))
    except Exception:
        return s


# ── Programme 1 — présences ──────────────────────────────────────────────────

def evaluate_presences(pred_xlsx, gt_xlsx):
    pred_rows = _load_sheet(pred_xlsx, 0)
    gt_rows   = _load_sheet(gt_xlsx,   0)
    if not pred_rows or not gt_rows:
        print("  [WARN] Feuille présences vide — ignorée")
        return {}

    gt_by_name = {str(r.get("imageName", r.get("IMAGE", ""))).strip(): r
                  for r in gt_rows}

    total = correct_grid = correct_sig = 0
    errors = []

    for row in pred_rows:
        name = str(row.get("imageName", "")).strip()
        gt = gt_by_name.get(name)
        if gt is None:
            continue
        total += 1
        p_grid = _norm(row.get("studentID_grid", ""))
        p_sig  = _norm(row.get("studentID_signature", ""))
        g_grid = _norm(gt.get("studentID_grid", gt.get("STUDENT_ID", "")))
        g_sig  = _norm(gt.get("studentID_signature", gt.get("STUDENT_ID_SIG", "")))

        ok_grid = p_grid == g_grid
        ok_sig  = p_sig  == g_sig
        if ok_grid: correct_grid += 1
        if ok_sig:  correct_sig  += 1
        if not ok_grid or not ok_sig:
            errors.append(f"      {name}: grille={p_grid}(GT={g_grid})  "
                          f"sig={p_sig}(GT={g_sig})")

    acc_g = correct_grid / total if total else 0
    acc_s = correct_sig  / total if total else 0
    print(f"  Présences — {total} images")
    print(f"    ID grille    : {correct_grid}/{total}  ({acc_g*100:.1f}%)")
    print(f"    ID signature : {correct_sig}/{total}  ({acc_s*100:.1f}%)")
    for e in errors:
        print(e)
    return {"n": total, "grid_acc": acc_g, "sig_acc": acc_s}


# ── Programme 2 — formulaires d'examen ───────────────────────────────────────

CHOICE_LABELS = list("ABCDEFGH")

PAGE1_FIELDS_TO_CHECK = [
    "Module", "Professor", "Date", "Code",
    "Notes de cours", "Notes manuscrites", "Ordinateur portable",
    "Calculatrice", "Feuilles brouillon",
    "Note maximale", "Note pour valider",
    "Prénom", "Nom", "Group", "STUDENT ID",
    "Validation signature", "Validation cryptogramme",
]


def evaluate_exam(pred_xlsx, gt_xlsx):
    print(f"  {Path(pred_xlsx).name}")

    # PAGE-01
    try:
        pred_p1 = {str(r.get("Field", "")).strip(): str(r.get("Value", "")).strip()
                   for r in _load_sheet(pred_xlsx, 0)}
        gt_p1   = {str(r.get("Field", "")).strip(): str(r.get("Value", "")).strip()
                   for r in _load_sheet(gt_xlsx,   0)}
    except Exception as e:
        print(f"    [WARN] PAGE-01 erreur: {e}")
        pred_p1, gt_p1 = {}, {}

    p1_total = p1_correct = 0
    p1_errors = []
    for field in PAGE1_FIELDS_TO_CHECK:
        if field not in gt_p1:
            continue
        p1_total += 1
        if _norm(pred_p1.get(field, "")) == _norm(gt_p1[field]):
            p1_correct += 1
        else:
            p1_errors.append(f"      {field}: prédit={pred_p1.get(field, '')!r}  "
                             f"GT={gt_p1[field]!r}")

    acc_p1 = p1_correct / p1_total if p1_total else 0
    print(f"    PAGE-01 : {p1_correct}/{p1_total} champs corrects ({acc_p1*100:.1f}%)")
    for e in p1_errors:
        print(e)

    # EXAM sheet
    try:
        pred_exam = _load_sheet(pred_xlsx, 1)
        gt_exam   = _load_sheet(gt_xlsx,   1)
    except Exception as e:
        print(f"    [WARN] Feuille EXAM erreur: {e}")
        return {"p1_acc": acc_p1, "choice_acc": 0.0, "q_detect_rate": 0.0}

    gt_by_q   = {_norm(r.get("QUESTION", "")): r for r in gt_exam   if r.get("QUESTION")}
    pred_by_q = {_norm(r.get("QUESTION", "")): r for r in pred_exam if r.get("QUESTION")}

    n_gt   = len(gt_by_q)
    n_pred = len(pred_by_q)
    q_rate = min(n_pred, n_gt) / n_gt if n_gt else 0

    choice_total = choice_correct = 0
    choice_errors = []

    for q, gt_row in gt_by_q.items():
        pred_row = pred_by_q.get(q)

        gt_choice   = next((c for c in CHOICE_LABELS
                            if _norm(gt_row.get(f"CHOIX {c}", "0")) == "1"), None)
        pred_choice = next((c for c in CHOICE_LABELS
                            if pred_row and _norm(pred_row.get(f"CHOIX {c}", "0")) == "1"),
                           None) if pred_row else None

        choice_total += 1
        if gt_choice == pred_choice:
            choice_correct += 1
        else:
            tag = "(non détectée)" if pred_row is None else ""
            choice_errors.append(f"      Q{q}: prédit={pred_choice!r}  GT={gt_choice!r} {tag}")

    acc_c = choice_correct / choice_total if choice_total else 0
    print(f"    EXAM : {n_pred} questions détectées (GT={n_gt}, taux={q_rate*100:.1f}%)")
    print(f"    CHOIX : {choice_correct}/{choice_total} corrects ({acc_c*100:.1f}%)")
    for e in choice_errors:
        print(e)

    return {"p1_acc": acc_p1, "q_detect_rate": q_rate, "choice_acc": acc_c}


# ── main ─────────────────────────────────────────────────────────────────────

def evaluate(results_dir, ground_truth_dir):
    results_dir      = Path(results_dir)
    ground_truth_dir = Path(ground_truth_dir)

    print("=" * 60)
    print("  ÉVALUATION QUANTITATIVE — DeepForm")
    print("=" * 60)

    all_grid = []
    all_sig  = []
    all_p1   = []
    all_q    = []
    all_c    = []

    # Programme 1
    pres_files = sorted(results_dir.glob("*PRESENCES*.xlsx"))
    if pres_files:
        print("\n── Programme 1 : Présences ──────────────────────────")
        for pred in pres_files:
            gt = ground_truth_dir / pred.name
            if not gt.exists():
                print(f"  [SKIP] GT introuvable : {pred.name}")
                continue
            m = evaluate_presences(pred, gt)
            if m:
                all_grid.append(m["grid_acc"])
                all_sig.append(m["sig_acc"])

    # Programme 2
    exam_files = sorted(f for f in results_dir.glob("*.xlsx")
                        if "PRESENCES" not in f.name)
    if exam_files:
        print("\n── Programme 2 : Formulaires d'examen ───────────────")
        for pred in exam_files:
            gt = ground_truth_dir / pred.name
            if not gt.exists():
                print(f"  [SKIP] GT introuvable : {pred.name}")
                continue
            m = evaluate_exam(pred, gt)
            all_p1.append(m.get("p1_acc", 0))
            all_q.append(m.get("q_detect_rate", 0))
            all_c.append(m.get("choice_acc", 0))
            print()

    # Synthèse
    def avg(lst):
        return f"{sum(lst)/len(lst)*100:.1f}%" if lst else "N/A"

    print("=" * 60)
    print("  SYNTHÈSE")
    print("=" * 60)
    if all_grid: print(f"  ID grille (prog.1)      : {avg(all_grid)}")
    if all_sig:  print(f"  ID signature (prog.1)   : {avg(all_sig)}")
    if all_p1:   print(f"  Champs PAGE-01 (prog.2) : {avg(all_p1)}")
    if all_q:    print(f"  Taux détection questions : {avg(all_q)}")
    if all_c:    print(f"  Précision choix A-H     : {avg(all_c)}")
    print("=" * 60)


if __name__ == "__main__":
    if len(sys.argv) < 3:
        print("Usage: python evaluate.py <results_dir> <ground_truth_dir>")
        sys.exit(1)
    evaluate(sys.argv[1], sys.argv[2])


## Étape 3 — Fournir les données

Deux options. **Choisissez-en une** et renseignez `DATA_ROOT`.

Structure attendue sous `DATA_ROOT` :
```
DATA_ROOT/
├── STUDENT_CLASS_SIGNATURES/      (base des signatures)
├── EXAM_FORM1_PRESENCES/          (photos 1ʳᵉ page)        ← Programme 1
├── EXAM_FORM1_PDF/                (PDF scannés, optionnel) ← Programme 2
├── EXAM_FORM2_PRESENCES/  ...
└── EXAM_FORM3_PRESENCES/  ...
```


La cellule de préparation ci-dessous accepte **deux dispositions** sans rien
changer :
- la disposition canonique ci-dessus, **ou**
- la disposition « brute » fournie pour le challenge :
  `FORM1/ FORM2/ FORM3/` (photos `.jpg`, PDF `.pdf` et vérités terrain `.xlsx`
  mélangés) + `SIGNATURES/` (archives `.zip`, une par lot).

Elle décompresse les signatures et range chaque `FORMx` en
`EXAM_FORMx_PRESENCES` / `EXAM_FORMx_PDF` / `EXAM_FORMx_GT`.


### Option A — Uploader un `.zip`

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()                       # sélectionnez votre .zip
zip_name = next(iter(up))
os.makedirs('/content/source', exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/source')
SOURCE = '/content/source'
print('Extrait dans', SOURCE)


### Option B — Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
print('Contenu de votre Drive :')
for it in sorted(os.listdir('/content/drive/MyDrive')):
    print('  -', it)
# Renseignez le dossier qui contient vos données (signatures + formulaires) :
SOURCE = '/content/drive/MyDrive/PROJECT 2026 -DATABASE-20260603'   # <-- ADAPTEZ
assert os.path.exists(SOURCE), f'Introuvable : {SOURCE}'
print('SOURCE =', SOURCE)


## Étape 4 — Préparer les données (disposition automatique)

Détecte la structure et construit `/content/data` au format attendu par les
programmes. Idempotent : ré-exécutable sans risque.


In [ ]:
import os, zipfile, shutil
from pathlib import Path

IMG_EXT = {".jpg", ".jpeg", ".png", ".heic", ".heif", ".webp", ".bmp", ".tif", ".tiff"}

def _find(root, name):
    root = Path(root)
    for p in [root, *root.rglob("*")]:
        if p.is_dir() and p.name == name:
            return p
    return None

def prepare_data(source):
    source = Path(source)
    work = Path("/content/data")
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True)

    # 1) Signatures — soit un dossier STUDENT_CLASS_SIGNATURES déjà prêt,
    #    soit un dossier SIGNATURES contenant des .zip à décompresser.
    sign = work / "STUDENT_CLASS_SIGNATURES"
    ready = _find(source, "STUDENT_CLASS_SIGNATURES")
    if ready is not None:
        shutil.copytree(ready, sign)
    else:
        sign.mkdir(parents=True)
        sdir = _find(source, "SIGNATURES")
        if sdir is not None:
            for z in sdir.glob("*.zip"):
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(sign)
    n_students = sum(1 for p in sign.iterdir() if p.is_dir())
    print(f"Signatures : {n_students} élèves dans {sign}")

    # 2) Formulaires — soit EXAM_FORMx_PRESENCES/_PDF déjà séparés,
    #    soit des dossiers FORM1/2/3 (ou EXAM_FORMx) à trier.
    canonical = sorted(p for p in source.rglob("EXAM_FORM*_PRESENCES") if p.is_dir())
    if canonical:
        for p in canonical:
            shutil.copytree(p, work / p.name, dirs_exist_ok=True)
        for p in source.rglob("EXAM_FORM*_PDF"):
            if p.is_dir():
                shutil.copytree(p, work / p.name, dirs_exist_ok=True)
    else:
        forms = sorted({p for p in [*source.iterdir(), *source.rglob("*")]
                        if p.is_dir() and "FORM" in p.name.upper()
                        and not p.name.endswith(("_PRESENCES", "_PDF", "_GT"))})
        seen = set()
        for fdir in forms:
            tag = "FORM" + "".join(c for c in fdir.name if c.isdigit())
            if tag in seen:
                continue
            seen.add(tag)
            pres = work / f"EXAM_{tag}_PRESENCES"; pres.mkdir(parents=True, exist_ok=True)
            pdfd = work / f"EXAM_{tag}_PDF";       pdfd.mkdir(parents=True, exist_ok=True)
            gt   = work / f"EXAM_{tag}_GT";        gt.mkdir(parents=True, exist_ok=True)
            for f in fdir.iterdir():
                ext = f.suffix.lower()
                if   ext in IMG_EXT: shutil.copy(f, pres / f.name)
                elif ext == ".pdf":  shutil.copy(f, pdfd / f.name)
                elif ext == ".xlsx": shutil.copy(f, gt / f.name)
            print(f"{tag}: {len(list(pres.iterdir()))} photos, "
                  f"{len(list(pdfd.iterdir()))} pdf, {len(list(gt.iterdir()))} GT")

    return str(work), str(sign)

DATA_ROOT, SIGNATURES_DIR = prepare_data(SOURCE)
print("\nDATA_ROOT      =", DATA_ROOT)
print("SIGNATURES_DIR =", SIGNATURES_DIR)
print("Contenu        :", sorted(p.name for p in Path(DATA_ROOT).iterdir()))


## Étape 5 — Programme 1 : validation des présences

Pour chaque formulaire disponible (`EXAM_FORMx_PRESENCES`), génère
`EXAM_FORMx_PRESENCES.xlsx` (colonnes `imageName`, `studentID_grid`,
`studentID_signature`).


In [ ]:
from pathlib import Path
from autoValidPresences import autoValidPresences

RESULTS_DIR = '/content/RESULTS'
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

for pres in sorted(Path(DATA_ROOT).glob('EXAM_FORM*_PRESENCES')):
    if pres.is_dir():
        print('\n==>', pres.name)
        autoValidPresences(str(pres), SIGNATURES_DIR, RESULTS_DIR)


## Étape 6 — Programme 2 : lecture automatique des formulaires PDF

Pour chaque `EXAM_FORMx_PDF`, génère un `.xlsx` par PDF (onglets `PAGE-01`
et `EXAM`). *(Peut être long : ~quelques secondes par page.)*


In [ ]:
from autoReadForm import autoReadForm

for pdfdir in sorted(Path(DATA_ROOT).glob('EXAM_FORM*_PDF')):
    if pdfdir.is_dir():
        print('\n==>', pdfdir.name)
        autoReadForm(str(pdfdir), SIGNATURES_DIR, RESULTS_DIR)


## Étape 7 — (Optionnel) Évaluation quantitative

Méthodologie §4.2 : apprentissage / validation / test sur FORM1 / FORM2 /
FORM3 avec vérité terrain lue depuis les noms de fichiers. Optimise le seuil
de décision des signatures et reporte la *balanced accuracy*.


In [ ]:
from optimize_threshold import collect_scores, optimise_threshold, _report

train = collect_scores(str(Path(DATA_ROOT)/'EXAM_FORM1_PRESENCES'), SIGNATURES_DIR)
val   = collect_scores(str(Path(DATA_ROOT)/'EXAM_FORM2_PRESENCES'), SIGNATURES_DIR)
test  = collect_scores(str(Path(DATA_ROOT)/'EXAM_FORM3_PRESENCES'), SIGNATURES_DIR)

t_train, acc = optimise_threshold(train)
t_val,   _   = optimise_threshold(val)
operating_t = round((t_train + t_val) / 2, 2)
print(f'Seuil opérationnel sélectionné = {operating_t:.2f}')
for name, recs in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    _report(name, recs, operating_t)


## Étape 8 — Télécharger les résultats

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/EXAM_RESULTS', 'zip', RESULTS_DIR)
files.download('/content/EXAM_RESULTS.zip')
